In [1]:
pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
CISB5123 Text Analytics - Lab Assignment 1
Student Name: ADIPUTRA BIN MOHD SUKI
Student ID: SW01083729
Student Name: MUHAMMAD 'AMMAR BIN JAILANI
Student ID: SW01083747

Task:
- Scrape Lazada Malaysia product reviews (limit 5 pages)
- Extract: reviewer name, review date, review content
- Save to CSV
"""

import time
import random
import requests
import pandas as pd


# Purpose: Build the Lazada review API URL for a given itemId and page number.
def build_review_url(item_id: str, page_no: int, page_size: int = 20) -> str:
    return (
        "https://my.lazada.com.my/pdp/review/getReviewList"
        f"?itemId={item_id}&pageSize={page_size}&filter=0&sort=0&pageNo={page_no}"
    )


# Purpose: Request one page of reviews from Lazada and return JSON safely.
def fetch_reviews_json(url: str, timeout: int = 20) -> dict:
    headers = {
        # Purpose: Add a common browser User-Agent to reduce blocking.
        "User-Agent": random.choice([
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0 Safari/537.36",
        ]),
        "Accept": "application/json, text/plain, */*",
    }

    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()  # Purpose: Stop if request failed (4xx/5xx)
    return resp.json()


# Purpose: Parse Lazada JSON into a list of rows (reviewer name, date, content).
def parse_reviews(data: dict) -> list[dict]:
    rows = []

    # Lazada response format usually: data["model"]["items"] is the list of reviews
    items = data.get("model", {}).get("items", [])
    for r in items:
        rows.append({
            "reviewer_name": r.get("buyerName", "").strip(),
            "review_date": r.get("reviewTime", "").strip(),
            "review_content": r.get("reviewContent", "").strip(),
        })

    return rows


# Purpose: Main function to scrape up to max_pages and save the output CSV.
def scrape_lazada_reviews(item_id: str, max_pages: int = 5, page_size: int = 20, sleep_seconds: float = 1.0) -> pd.DataFrame:
    all_rows = []

    for page_no in range(1, max_pages + 1):
        url = build_review_url(item_id=item_id, page_no=page_no, page_size=page_size)
        print(f"Fetching page {page_no}: {url}")

        try:
            data = fetch_reviews_json(url)
            page_rows = parse_reviews(data)

            if not page_rows:
                print(f"No reviews found on page {page_no}. Stopping early.")
                break

            all_rows.extend(page_rows)
            print(f"Collected {len(page_rows)} reviews from page {page_no}. Total now: {len(all_rows)}")

        except requests.RequestException as e:
            print(f"Request failed on page {page_no}: {e}")
            # You can choose to break or continue; I continue after a pause
        except ValueError as e:
            print(f"JSON parsing failed on page {page_no}: {e}")

        time.sleep(sleep_seconds)

    df = pd.DataFrame(all_rows)
    return df


# ---- RUN HERE ----
ITEM_ID = "4175864282"  # from your link
MAX_PAGES = 5

df = scrape_lazada_reviews(item_id=ITEM_ID, max_pages=MAX_PAGES, page_size=20, sleep_seconds=1.0)

output_csv = f"lazada_reviews_item_{ITEM_ID}.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("\nSaved:", output_csv)
print("Preview:")
display(df.head(10) if len(df) else df)

Fetching page 1: https://my.lazada.com.my/pdp/review/getReviewList?itemId=4175864282&pageSize=20&filter=0&sort=0&pageNo=1
Collected 20 reviews from page 1. Total now: 20
Fetching page 2: https://my.lazada.com.my/pdp/review/getReviewList?itemId=4175864282&pageSize=20&filter=0&sort=0&pageNo=2
Request failed on page 2: Expecting value: line 2 column 1 (char 2)
Fetching page 3: https://my.lazada.com.my/pdp/review/getReviewList?itemId=4175864282&pageSize=20&filter=0&sort=0&pageNo=3
Request failed on page 3: Expecting value: line 2 column 1 (char 2)
Fetching page 4: https://my.lazada.com.my/pdp/review/getReviewList?itemId=4175864282&pageSize=20&filter=0&sort=0&pageNo=4
Request failed on page 4: Expecting value: line 2 column 1 (char 2)
Fetching page 5: https://my.lazada.com.my/pdp/review/getReviewList?itemId=4175864282&pageSize=20&filter=0&sort=0&pageNo=5
Request failed on page 5: Expecting value: line 2 column 1 (char 2)

Saved: lazada_reviews_item_4175864282.csv
Preview:


,reviewer_name,review_date,review_content
0,siti zubaidah,01 Jul 2025,brang sampai dgan selamat.. cuma nye ketat.. p...
1,Kenneth Lee,13 Jun 2025,"Great for casual wear, Stylish and sporty, Tha..."
2,c***u,15 Jun 2025,cantik gila hoiiiii .....boleh la beli lagi❤️❤...
3,Manish,18 Jul 2025,comfortable
4,yeo lian heng,28 Jun 2025,"Fits true to size,"
5,Mahmud Abd Wahab,31 Dec 2024,"Fits true to size, the size is perfect as orde..."
6,T***r,13 Dec 2024,"Sangat baik dan kelihatan sangat bagus,👍\nআমার..."
7,limrambo,19 Jun 2025,"Stylish and sporty,"
8,c***t,02 Apr 2025,"Fits true to size, Trendy design,"
9,Kacey Tan,13 Mar 2025,"Great for casual wear, Trendy design,"
